# 🔍 Embryoscope Silver Layer Reconciliation: Local DuckDB vs AWS Athena Production

This notebook automates the data quality reconciliation between the local **DuckDB** database (`huntington_data_lake.duckdb`, schema `silver_embryoscope`) and the production **AWS Athena** database (`silver_embryoscope_prod`).

### Objectives:
1. **Row Count Audit**: Compare total records per table.
2. **Primary Key Overlap Audit**: Reconcile exact keys present in both environments, only locally, or only in production.
3. **Unit/Location Breakdown**: Group counts by clinic server to check for isolated sync failures.
4. **Yearly Breakdown**: Drill down row counts and key matches per calendar year (using mapped date columns).
5. **Newest Record Analysis**: Fetch and show the most recent records present in only one environment (ordered by primary key descending).

### Database Connections:
- **Local**: DuckDB (`huntington_data_lake.duckdb` -> schema: `silver_embryoscope`)
- **AWS Athena**: PyAthena (`silver_embryoscope_prod` database)

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Configuration
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_embryoscope_prod'

print("Libraries imported. Configured database paths:")
print(f"  Local DuckDB: {os.path.abspath(DUCKDB_PATH)}")
print(f"  AWS Athena:   {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")

## 🔌 Connection Helpers
Defining wrapper functions to connect, execute queries, and guarantee connection closure.

In [ ]:
def run_duck(query):
    """Runs a query on local DuckDB, ensuring connection is closed."""
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        return conn.execute(query).df()
    finally:
        conn.close()

def run_athena(query):
    """Runs a query on AWS Athena, ensuring connection is closed."""
    conn = connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB)
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

def get_common_columns(table):
    # 1. Local columns
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        local_cols = set([c[0].lower() for c in conn.execute(f"SELECT * FROM silver_embryoscope.{table} LIMIT 0").description])
    finally:
        conn.close()
    
    # 2. Athena columns
    try:
        prod_df = run_athena(f"SELECT * FROM silver_embryoscope_prod.{table} LIMIT 0")
        prod_cols = set([c.lower() for c in prod_df.columns])
    except Exception:
        prod_cols = set()
        
    return list(local_cols & prod_cols)

# Test connections
try:
    duck_ok = run_duck("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ DuckDB Connection: OK")
except Exception as e:
    print(f"❌ DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    athena_ok = run_athena("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False

## 🗺️ Configuration & Predefined Mappings
Defining primary keys, date columns, and location/server fields.

In [ ]:
TABLES_CONFIG = {
    "patients": {
        "local_table": "patients",
        "athena_table": "patients",
        "keys": ["PatientIDx"],
        "target_keys": ["patient_id_x"],
        "date_col": "DateOfBirth",
        "target_date_col": "date_of_birth",
        "local_loc_col": "_location",
        "target_loc_col": "source_server"
    },
    "treatments": {
        "local_table": "treatments",
        "athena_table": "treatments",
        "keys": ["PatientIDx", "TreatmentName"],
        "target_keys": ["patient_id_x", "treatment_name"],
        "date_col": "_extraction_timestamp",
        "target_date_col": "bronze_updated_at",
        "local_loc_col": "unit_huntington",
        "target_loc_col": "source_server"
    },
    "embryo_data": {
        "local_table": "embryo_data",
        "athena_table": "embryo_data",
        "keys": ["EmbryoID"],
        "target_keys": ["embryo_id"],
        "date_col": "KIDDate",
        "target_date_col": "kid_date",
        "local_loc_col": "unit_huntington",
        "target_loc_col": "unit_huntington"
    }
}
print("Configured key, date, and location mappings for Embryoscope tables.")

## 📊 Part 1: Row Count & Key Reconciliation Summary
Iterating through all configured tables to compare total rows, match counts, and overlaps.

In [ ]:
def normalize_key(val):
    if val is None or pd.isna(val):
        return None
    val_str = str(val).strip().lower()
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    if val_str.isdigit():
        try:
            val_str = str(int(val_str))
        except ValueError:
            pass
    if " 00:00:00" in val_str:
        val_str = val_str.replace(" 00:00:00", "")
    return val_str

summary_data = []
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    
    l_select_sql = ", ".join([f'"{c}"' for c in l_keys])
    t_select_sql = ", ".join([f'"{c}"' for c in t_keys])
    
    local_count = run_duck(f"SELECT COUNT(*) as cnt FROM {local_tbl}").iloc[0]['cnt']
    prod_count = run_athena(f"SELECT COUNT(*) as cnt FROM {target_tbl}").iloc[0]['cnt']
    
    local_df = run_duck(f"SELECT {l_select_sql} FROM {local_tbl}")
    local_df.columns = [c.lower() for c in local_df.columns]
    
    prod_df = run_athena(f"SELECT {t_select_sql} FROM {target_tbl}")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    
    l_keys_lower = [k.lower() for k in l_keys]
    t_keys_lower = [tk.lower() for tk in t_keys]
    
    local_df['_comp_key'] = local_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in l_keys_lower]), axis=1
    )
    prod_df['_comp_key'] = prod_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in t_keys_lower]), axis=1
    )
    
    local_keys_set = set(local_df['_comp_key'].dropna().tolist())
    prod_keys_set = set(prod_df['_comp_key'].dropna().tolist())
    
    matched_keys = len(local_keys_set & prod_keys_set)
    only_local = len(local_keys_set - prod_keys_set)
    only_prod = len(prod_keys_set - local_keys_set)
    
    summary_data.append({
        'Table': t,
        'Local Rows': local_count,
        'Athena Rows': prod_count,
        'Difference': local_count - prod_count,
        'Match Count': matched_keys,
        'Only in Local (DuckDB)': only_local,
        'Only in Prod (Athena)': only_prod
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    'Local Rows': '{:,}',
    'Athena Rows': '{:,}',
    'Difference': '{:+,}',
    'Match Count': '{:,}',
    'Only in Local (DuckDB)': '{:,}',
    'Only in Prod (Athena)': '{:,}'
}).bar(subset=['Difference'], align='mid', color=['#d65f5f', '#5fba7d'])

## 🏥 Part 1.2: Location/Unit Breakdown
Grouping records by location server to pinpoint which clinics are out of sync.

In [ ]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    loc_col = cfg['local_loc_col']
    t_loc_col = cfg['target_loc_col']
    
    print("=" * 80)
    print(f"🏥 Table: {t} | Unit Breakdown ({loc_col} vs {t_loc_col})")
    print("=" * 80)
    
    duck_loc = run_duck(f"SELECT \"{loc_col}\" as unit, COUNT(*) as local_cnt FROM {local_tbl} GROUP BY 1")
    ath_loc = run_athena(f"SELECT \"{t_loc_col}\" as unit, COUNT(*) as athena_cnt FROM {target_tbl} GROUP BY 1")
    
    merged_loc = pd.merge(duck_loc, ath_loc, on='unit', how='outer').fillna(0)
    merged_loc['local_cnt'] = merged_loc['local_cnt'].astype(int)
    merged_loc['athena_cnt'] = merged_loc['athena_cnt'].astype(int)
    merged_loc['diff'] = merged_loc['local_cnt'] - merged_loc['athena_cnt']
    
    display(merged_loc)
    print("\n")

## 📅 Part 2: Yearly Breakdown Analysis
Drilling down row counts and key overlaps per year for each table using date/timestamp columns.

In [ ]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    date_col = cfg['date_col']
    target_date_col = cfg['target_date_col']
    
    l_keys_lower = [k.lower() for k in l_keys]
    t_keys_lower = [tk.lower() for tk in t_keys]
    
    print("=" * 80)
    print(f"📊 Table: {t} (Local Date: {date_col} | Athena Date: {target_date_col})")
    print("=" * 80)
    
    l_select_sql = ", ".join([f'"{c}"' for c in l_keys + [date_col]])
    t_select_sql = ", ".join([f'"{c}"' for c in t_keys + [target_date_col]])
    
    local_df = run_duck(f"SELECT {l_select_sql} FROM {local_tbl}")
    local_df.columns = [c.lower() for c in local_df.columns]
    
    prod_df = run_athena(f"SELECT {t_select_sql} FROM {target_tbl}")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    
    local_df['_comp_key'] = local_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in l_keys_lower]), axis=1
    )
    prod_df['_comp_key'] = prod_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in t_keys_lower]), axis=1
    )
    
    # Safely convert to datetime and extract year
    local_df['record_year'] = pd.to_datetime(local_df[date_col.lower()], errors='coerce').dt.year
    local_df['record_year'] = local_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    
    prod_df['record_year'] = pd.to_datetime(prod_df[target_date_col.lower()], errors='coerce').dt.year
    prod_df['record_year'] = prod_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    
    years = sorted(list(set(local_df['record_year'].dropna().tolist()) | set(prod_df['record_year'].dropna().tolist())))
    
    yearly_summary = []
    for yr in years:
        l_keys_set = set(local_df[local_df['record_year'] == yr]['_comp_key'].dropna().tolist())
        p_keys_set = set(prod_df[prod_df['record_year'] == yr]['_comp_key'].dropna().tolist())
        
        matched = len(l_keys_set & p_keys_set)
        only_l = len(l_keys_set - p_keys_set)
        only_p = len(p_keys_set - l_keys_set)
        
        yearly_summary.append({
            'Year': yr,
            'Local Count': len(l_keys_set),
            'Athena Count': len(p_keys_set),
            'Matched Count': matched,
            'Only Local': only_l,
            'Only Athena': only_p
        })
    
    yearly_df = pd.DataFrame(yearly_summary)
    display(yearly_df)
    print("\n")

## 🔬 Part 2.2: Zoom-In Ibirapuera Yearly Breakdown
Reconciling years exclusively for the **Ibirapuera** unit where the discrepancies are concentrated.

In [ ]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    loc_col = cfg['local_loc_col']
    t_loc_col = cfg['target_loc_col']
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    date_col = cfg['date_col']
    target_date_col = cfg['target_date_col']
    
    print("=" * 80)
    print(f"🔍 Table: {t} | Ibirapuera Only Yearly Breakdown")
    print("=" * 80)
    
    l_select_sql = ", ".join([f'"{c}"' for c in l_keys + [date_col]])
    t_select_sql = ", ".join([f'"{c}"' for c in t_keys + [target_date_col]])
    
    local_df = run_duck(f"SELECT {l_select_sql} FROM {local_tbl} WHERE \"{loc_col}\" = 'Ibirapuera'")
    local_df.columns = [c.lower() for c in local_df.columns]
    
    prod_df = run_athena(f"SELECT {t_select_sql} FROM {target_tbl} WHERE \"{t_loc_col}\" = 'Ibirapuera'")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    
    local_df['_comp_key'] = local_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in [k.lower() for k in l_keys]]), axis=1
    )
    prod_df['_comp_key'] = prod_df.apply(
        lambda r: "||".join([normalize_key(r[k]) if normalize_key(r[k]) is not None else "NULL" for k in [tk.lower() for tk in t_keys]]), axis=1
    )
    
    local_df['record_year'] = pd.to_datetime(local_df[date_col.lower()], errors='coerce').dt.year
    local_df['record_year'] = local_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    
    prod_df['record_year'] = pd.to_datetime(prod_df[target_date_col.lower()], errors='coerce').dt.year
    prod_df['record_year'] = prod_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    
    years = sorted(list(set(local_df['record_year'].dropna().tolist()) | set(prod_df['record_year'].dropna().tolist())))
    
    yearly_summary = []
    for yr in years:
        l_keys_set = set(local_df[local_df['record_year'] == yr]['_comp_key'].dropna().tolist())
        p_keys_set = set(prod_df[prod_df['record_year'] == yr]['_comp_key'].dropna().tolist())
        
        matched = len(l_keys_set & p_keys_set)
        only_l = len(l_keys_set - p_keys_set)
        only_p = len(p_keys_set - l_keys_set)
        
        yearly_summary.append({
            'Year': yr,
            'Local Count': len(l_keys_set),
            'Athena Count': len(p_keys_set),
            'Matched Count': matched,
            'Only Local': only_l,
            'Only Athena': only_p
        })
    
    display(pd.DataFrame(yearly_summary))
    print("\n")

## 🆕 Part 3: Mismatch Drill-Down — Newest Mismatched Records
Displaying up to 5 newest records unique to DuckDB or Athena for debugging purposes.

In [ ]:
for t, cfg in TABLES_CONFIG.items():
    local_tbl = f"silver_embryoscope.{cfg['local_table']}"
    target_tbl = f"silver_embryoscope_prod.{cfg['athena_table']}"
    l_keys = cfg['keys']
    t_keys = cfg['target_keys']
    key = l_keys[0]
    t_key = t_keys[0]
    
    l_keys_lower = [k.lower() for k in l_keys]
    t_keys_lower = [tk.lower() for tk in t_keys]
    
    local_df = run_duck(f"SELECT \"{key}\" FROM {local_tbl}")
    local_df.columns = [c.lower() for c in local_df.columns]
    local_keys_set = set(local_df[key.lower()].dropna().tolist())
    
    prod_df = run_athena(f"SELECT \"{t_key}\" FROM {target_tbl}")
    prod_df.columns = [c.lower() for c in prod_df.columns]
    prod_keys_set = set(prod_df[t_key.lower()].dropna().tolist())
    
    only_l = sorted(list(local_keys_set - prod_keys_set), reverse=True)
    only_p = sorted(list(prod_keys_set - local_keys_set), reverse=True)
    
    print("=" * 80)
    print(f"🔍 Mismatch Samples: {t} (Primary Key: {key})")
    print("=" * 80)
    
    if only_l:
        sample_keys = only_l[:5]
        keys_ph = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        local_samples = run_duck(f"SELECT * FROM {local_tbl} WHERE \"{key}\" IN ({keys_ph}) ORDER BY \"{key}\" DESC")
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Local DuckDB (Total: {len(only_l)}):")
        display(local_samples)
    else:
        print("✅ No records found exclusively in Local DuckDB.")
        
    if only_p:
        sample_keys = only_p[:5]
        keys_ph = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        prod_samples = run_athena(f"SELECT * FROM {target_tbl} WHERE \"{t_key}\" IN ({keys_ph}) ORDER BY \"{t_key}\" DESC")
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Athena Production (Total: {len(only_p)}):")
        display(prod_samples)
    else:
        print("✅ No records found exclusively in Athena Production.")
    print("\n")